#Load bronze.crm_cust_info, tansform it and write to the silver schema 

In [0]:
# import libraries
from pyspark.sql.window import Window
from pyspark.sql import functions as F

## Step 1: Load bronze.crm_cust_info:

In [0]:
# Load bronze.crm_cust_info table from bronze schema to a dataframe
df = spark.read.table("data_lakehouse.bronze.crm_cust_info")
df.display()

## Step 2: Clean and Transform the data

In [0]:
# copy the original dataframe to a new dataframe
df_clean = df

In [0]:
# check number of duplicates
df_clean.groupBy("cst_key").count().filter("count > 1").display()

In [0]:
# Remove duplicate rows based on cst_key, but retain the row with the latest date, using window function
w = Window.partitionBy("cst_key").orderBy(df_clean.cst_create_date.desc())
df_clean = df_clean.withColumn("row_num", F.row_number().over(w)).filter("row_num = 1").drop("row_num")


In [0]:
# rename all column names using for loop

columns = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

for old, new in columns.items():
  df_clean = df_clean.withColumnRenamed(old, new)

df_clean.columns 

In [0]:
# select all rows where customer_id is null
df_clean.filter(df_clean.customer_id.isNull()).display()

In [0]:
# delete all rows where customer_id is null
df_clean = df_clean.filter(df_clean.customer_id.isNotNull())

In [0]:
# check each column and count the number of null values
for column in df_clean.columns:
    print(f"{column}: {df_clean.filter(F.col(column).isNull()).count()}")



In [0]:
#check distint values in the 'gender' and marital_status column
df_clean.groupBy("gender").count().display()
df_clean.groupBy("marital_status").count().display()

In [0]:
# Replace the null values in the 'gender' column with 'Unknown' and 'M' and 'F' with 'Male' and 'Female' respectively
df_clean = df_clean.withColumn("gender", 
                               F.when(F.col("gender").isNull(), "Unknown")
                               .when(F.col("gender") == "M", "Male")
                               .when(F.col("gender") == "F", "Female")
                               .otherwise(F.col("gender")))


In [0]:
# Replace the 'M' and 'S' in the 'marital_status' column with 'Married' and 'Single' respectively.

df_clean = df_clean.withColumn("marital_status", 
                               F.when(F.col("marital_status") == "M", "Married")
                               .when(F.col("marital_status") == "S", "Single")
                               .otherwise(F.col("marital_status")))


In [0]:
# Remove all white spaces in each column
df_clean = df_clean.select([F.trim(F.col(c)).alias(c) for c in df_clean.columns])


## Step 3: Write the dataframe to the silver schema

In [0]:
# Write the cleaned dataframe to the silver schema

df_clean.write.saveAsTable(name="data_lakehouse.silver.silver_crm_cust_info", mode="overwrite", format="delta")


In [0]:
/Workspace/Users/petermbegbu@gmail.com/data_lakehouse_project